# Walidacja liniowego fitu Lyα $P_{1D}$

Ten notebook wykonuje niezależną walidację benchmarku `lcdm_2021_massless` z pracy arXiv:2011.03050. Jego celem jest:

1. sprawdzić pliki danych i integralność zapisanego pliku teorii;
2. ponownie obliczyć $\chi^2$ dla zapisanego best fitu;
3. opcjonalnie wykonać pełny, wielostartowy fit od początku;
4. porównać wynik z wartością publikowaną $\chi^2_{\rm linear}=206.5$;
5. narysować dane, best fit, pullsy i diagnostyki według redshiftu.

> Notebook nie uruchamia CLASS. Plik `smoke.npz` jest odpowiedni dla benchmarku liniowego, ponieważ tryb `linear` korzysta z gęstej tablicy `p_total_input`. Zapisanych w nim kanałów one-loop nie interpretujemy tutaj naukowo.


## 1. Instalacja i sposób uruchomienia

Najwygodniej uruchomić terminal w głównym katalogu projektu i jednorazowo wykonać:

```bash
python -m pip install -e .
python -m pip install pandas matplotlib jupyter
```

Instalacja `-e` jest zalecana, ale nie obowiązkowa: poniższa komórka potrafi również znaleźć lokalny folder projektu i dodać go do `sys.path`. Następnie wybierz **Run All**.


In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value.to_string(index=False) if hasattr(value, "to_string") else value)

pd.set_option("display.max_columns", None)
plt.style.use("seaborn-v0_8-whitegrid")

# Działa zarówno po uruchomieniu notebooka z katalogu projektu,
# jak i bezpośrednio z katalogu notebooks/.
candidates = [
    Path.cwd(),
    Path.cwd() / "lyalpha_one_loop",
    Path.cwd().parent,
    Path.cwd().parent / "lyalpha_one_loop",
]
PROJECT_ROOT = next(
    (path.resolve() for path in candidates if (path / "pyproject.toml").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Nie znaleziono głównego katalogu projektu. Uruchom notebook z folderu "
        "lyalpha_one_loop lub zmień listę candidates."
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from lyalpha_pt.data import load_dr12
from lyalpha_pt.fit import EffectiveModelFit, PARAMETER_NAMES, write_fit_result
from lyalpha_pt.theory import load_theory

print("PROJECT_ROOT =", PROJECT_ROOT)
print("Python =", sys.version.split()[0])


## 2. Konfiguracja walidacji

Przy pierwszej walidacji pozostaw `RUN_NEW_FIT = True`. Później można ustawić `False`, aby szybko odtwarzać wykresy z zapisanego best fitu.


In [ ]:
DATA_DIR = PROJECT_ROOT / "data"
THEORY_PATH = PROJECT_ROOT / "results" / "lcdm_2021_massless_smoke.npz"
SAVED_FIT_PATH = PROJECT_ROOT / "results" / "lcdm_2021_massless_linear_fit.json"
OUTPUT_DIR = PROJECT_ROOT / "results" / "linear_validation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODE = "linear"
COVARIANCE_MODE = "paper_diag"
K_UV_CUT = 20.0  # h/Mpc

RUN_NEW_FIT = True
FIT_SEEDS = [12345, 23456, 34567]
DE_MAXITER = 300
DE_POPSIZE = 20

PUBLISHED_CHI2 = 206.5
PUBLISHED_TOLERANCE = 0.05
REPRODUCTION_TOLERANCE = 1.0e-5
MULTISTART_TOLERANCE = 1.0e-4
SAVE_FIGURES = True

print("Theory:", THEORY_PATH)
print("Data:  ", DATA_DIR)
print("Output:", OUTPUT_DIR)


## 3. Wczytanie i kontrola danych oraz teorii

`load_dr12` wymaga pełnego publicznego wydania: 13 bloków redshiftu po 35 punktów. Do fitu wybierane są automatycznie bloki $z=3.0,\ldots,4.2$, czyli 245 punktów. `load_theory` sprawdza digest SHA-256 pliku teorii.


In [ ]:
dataset = load_dr12(DATA_DIR)
theory = load_theory(THEORY_PATH)  # automatycznie sprawdza digest

fitter = EffectiveModelFit(
    dataset,
    theory,
    mode=MODE,
    k_uv_cut=K_UV_CUT,
    covariance_mode=COVARIANCE_MODE,
)

input_table = pd.DataFrame({
    "quantity": [
        "selected data points",
        "selected redshift bins",
        "k_v min [s/km]",
        "k_v max [s/km]",
        "theory model",
        "linear k-grid points",
        "linear k max [h/Mpc]",
        "bundle digest",
    ],
    "value": [
        dataset.n_data,
        len(dataset.z_unique),
        float(dataset.k_velocity.min()),
        float(dataset.k_velocity.max()),
        theory.metadata["model"]["name"],
        len(theory.k_input),
        float(theory.k_input[-1]),
        theory.metadata["bundle_digest"],
    ],
})
display(input_table)

assert dataset.n_data == 245
assert np.allclose(dataset.z_unique, np.arange(3.0, 4.21, 0.2))
assert np.allclose(dataset.z_unique, theory.z)


## 4. Reprodukcja zapisanego best fitu

JSON przechowuje parametry, ale nie gotową krzywą. Poniżej ponownie konstruujemy $P_{1D}^{\rm model}$ z pliku teorii i sprawdzamy, czy otrzymujemy dokładnie zapisane $\chi^2$.


In [ ]:
with SAVED_FIT_PATH.open(encoding="utf-8") as handle:
    saved_output = json.load(handle)

saved_record = saved_output["fits"]["linear"]
theta_saved = np.array([
    saved_record["parameters"][name] for name in PARAMETER_NAMES
])
saved_chi2_declared = float(saved_record["chi2"])
saved_chi2_recomputed = fitter.chi2(theta_saved)

saved_check = pd.DataFrame({
    "quantity": [
        "chi2 declared in JSON",
        "chi2 recomputed from data + theory",
        "absolute difference",
    ],
    "value": [
        saved_chi2_declared,
        saved_chi2_recomputed,
        abs(saved_chi2_declared - saved_chi2_recomputed),
    ],
})
display(saved_check)

assert saved_record["theory_digest"] == theory.metadata["bundle_digest"]
assert abs(saved_chi2_declared - saved_chi2_recomputed) < REPRODUCTION_TOLERANCE


## 5. Niezależny wielostartowy fit

Każdy seed uruchamia osobną optymalizację globalną, a następnie lokalne polerowanie Powella. Zgodność minimów jest istotniejsza niż sama flaga `success`. Przy domyślnych ustawieniach trzy uruchomienia powinny zająć około kilkudziesięciu sekund na zwykłym komputerze.


In [ ]:
fit_runs = []
bounds = fitter.bounds_broad()

if RUN_NEW_FIT:
    for seed in FIT_SEEDS:
        start = time.perf_counter()
        result = fitter.fit_once(
            seed=seed,
            bounds=bounds,
            de_maxiter=DE_MAXITER,
            de_popsize=DE_POPSIZE,
        )
        elapsed = time.perf_counter() - start
        fit_runs.append({
            "seed": seed,
            "result": result,
            "chi2": fitter.chi2(result.x),
            "success": bool(result.success),
            "elapsed_s": elapsed,
        })
        print(
            f"seed={seed}: chi2={fit_runs[-1]['chi2']:.9f}, "
            f"success={result.success}, time={elapsed:.1f} s"
        )

    best_run = min(fit_runs, key=lambda item: item["chi2"])
    best_result = best_run["result"]
    theta_best = np.asarray(best_result.x, dtype=float)
else:
    best_result = None
    theta_best = theta_saved.copy()

multistart_table = pd.DataFrame([
    {
        "seed": item["seed"],
        "chi2": item["chi2"],
        "success": item["success"],
        "elapsed_s": item["elapsed_s"],
    }
    for item in fit_runs
])
if not multistart_table.empty:
    display(multistart_table)


## 6. Automatyczny test akceptacyjny

Wynik uznajemy za odtworzony, jeżeli:

- ponownie obliczone $\chi^2$ zgadza się z JSON;
- nowy fit zgadza się z zapisanym minimum;
- wynik leży w odległości mniejszej niż 0.05 od wartości publikowanej 206.5;
- różne seedy znajdują to samo minimum;
- żaden parametr nie leży w odległości mniejszej niż 1% szerokości boxu od granicy.


In [ ]:
chi2_best = fitter.chi2(theta_best)
summary = fitter.summary(theta_best)
boundary_table = pd.DataFrame(fitter.boundary_diagnostics(theta_best, bounds))

if RUN_NEW_FIT:
    multistart_spread = float(multistart_table["chi2"].max() - multistart_table["chi2"].min())
else:
    multistart_spread = 0.0

validation_table = pd.DataFrame([
    {
        "test": "JSON reproduction",
        "value": abs(saved_chi2_declared - saved_chi2_recomputed),
        "tolerance": REPRODUCTION_TOLERANCE,
        "pass": abs(saved_chi2_declared - saved_chi2_recomputed) < REPRODUCTION_TOLERANCE,
    },
    {
        "test": "new fit vs saved fit",
        "value": abs(chi2_best - saved_chi2_declared),
        "tolerance": REPRODUCTION_TOLERANCE,
        "pass": abs(chi2_best - saved_chi2_declared) < REPRODUCTION_TOLERANCE,
    },
    {
        "test": "new fit vs published 206.5",
        "value": abs(chi2_best - PUBLISHED_CHI2),
        "tolerance": PUBLISHED_TOLERANCE,
        "pass": abs(chi2_best - PUBLISHED_CHI2) < PUBLISHED_TOLERANCE,
    },
    {
        "test": "multistart spread",
        "value": multistart_spread,
        "tolerance": MULTISTART_TOLERANCE,
        "pass": multistart_spread < MULTISTART_TOLERANCE,
    },
    {
        "test": "no parameter near bound",
        "value": int(boundary_table["near_bound"].sum()),
        "tolerance": 0,
        "pass": not boundary_table["near_bound"].any(),
    },
])
display(validation_table)

VALIDATION_PASSED = bool(validation_table["pass"].all())
print("VALIDATION:", "PASSED" if VALIDATION_PASSED else "FAILED")
print(f"chi2 = {chi2_best:.9f}")
print(f"chi2/dof = {summary['chi2_per_dof']:.6f}")
print(f"p-value = {summary['p_value']:.6f}")
assert VALIDATION_PASSED, "Co najmniej jeden test walidacyjny nie przeszedł."


## 7. Parametry best fit i diagnostyka granic

Duża wartość $\beta_{\rm ct}$ nie oznacza automatycznie problemu. Istotne jest, czy minimum nie znajduje się na granicy oraz czy po otrzymaniu teorii one-loop wynik pozostanie stabilny przy zmianie $k_{\rm UV}$.


In [ ]:
parameter_table = boundary_table.copy()
parameter_table["saved_value"] = theta_saved
parameter_table["new_minus_saved"] = theta_best - theta_saved
display(parameter_table)
print(f"alpha_F = exp(log_alpha_F) = {np.exp(theta_best[0]):.10f}")


## 8. Predykcja, pullsy i $\chi^2$ według redshiftu

Dla `paper_diag` pull jest równy $(P_{\rm model}-P_{\rm data})/\sigma_{\rm total}$. Suma kwadratów pullsów musi dać całkowite $\chi^2$. Dla kowariancji niediagonalnej należałoby zamiast tego analizować residua wybielone.


In [ ]:
model_best = fitter.model(theta_best)
residual = model_best - dataset.p1d
pull = residual / dataset.sigma_total

chi2_from_pulls = float(np.sum(pull**2))
assert abs(chi2_from_pulls - chi2_best) < 1.0e-8

chi2_by_z = pd.DataFrame(fitter.chi2_by_redshift(theta_best))
pull_rows = []
for z in dataset.z_unique:
    mask = np.isclose(dataset.z, z)
    pull_rows.append({
        "z": z,
        "mean_pull": float(np.mean(pull[mask])),
        "rms_pull": float(np.sqrt(np.mean(pull[mask]**2))),
        "max_abs_pull": float(np.max(np.abs(pull[mask]))),
        "n_abs_pull_gt_2": int(np.count_nonzero(np.abs(pull[mask]) > 2)),
        "n_abs_pull_gt_3": int(np.count_nonzero(np.abs(pull[mask]) > 3)),
    })
pull_table = pd.DataFrame(pull_rows)
chi2_diagnostics = chi2_by_z.merge(pull_table, on="z")
display(chi2_diagnostics)
print("sum chi2(z) =", chi2_by_z["chi2"].sum())


## 9. Dane i liniowy best fit — wszystkie redshifty

Pokazujemy konwencję $k_vP_{1D}(k_v)/\pi$, używaną w naszych wcześniejszych porównaniach. Punkty i błędy pochodzą bezpośrednio z publicznych plików danych; linia jest rzeczywistą predykcją best fitu bez wizualnych korekt.


In [ ]:
def plot_quantity(kv, p1d):
    return kv * p1d / np.pi

z_values = dataset.z_unique
ncols = 2
nrows = int(np.ceil(len(z_values) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(13, 4.0 * nrows))
axes = np.asarray(axes).ravel()

for ax, z in zip(axes, z_values):
    mask = np.isclose(dataset.z, z)
    order = np.argsort(dataset.k_velocity[mask])
    kv = dataset.k_velocity[mask][order]
    data_y = plot_quantity(kv, dataset.p1d[mask][order])
    error_y = plot_quantity(kv, dataset.sigma_total[mask][order])
    model_y = plot_quantity(kv, model_best[mask][order])
    chi2_z = chi2_by_z.loc[np.isclose(chi2_by_z["z"], z), "chi2"].iloc[0]

    ax.errorbar(
        kv, data_y, yerr=error_y, fmt="o", ms=3.8, capsize=2,
        color="black", ecolor="0.55", label="BOSS data",
    )
    ax.plot(kv, model_y, lw=2.1, color="tab:blue", label="linear best fit")
    ax.set_xscale("log")
    ax.set_xlabel(r"$k_v\,[\mathrm{s/km}]$")
    ax.set_ylabel(r"$k_vP_{1D}(k_v)/\pi$")
    ax.set_title(rf"$z={z:.1f}$, $\chi^2_z={chi2_z:.2f}$")
    ax.grid(alpha=0.25)

for ax in axes[len(z_values):]:
    ax.set_visible(False)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles, labels, loc="upper center", bbox_to_anchor=(0.5, 0.995),
    ncol=2, frameon=False,
)
fig.suptitle(
    rf"Linear LCDM benchmark: $\chi^2={chi2_best:.3f}$, "
    rf"$\chi^2/\mathrm{{dof}}={summary['chi2_per_dof']:.3f}$",
    y=1.015,
)
fig.tight_layout(rect=[0, 0, 1, 0.955])
if SAVE_FIGURES:
    fig.savefig(OUTPUT_DIR / "linear_best_fit_panels.png", dpi=180, bbox_inches="tight")
plt.show()


## 10. Pullsy według redshiftu

Systematyczny trend pullsów z $k_v$ byłby bardziej niepokojący niż pojedynczy odległy punkt. Zacieniowany obszar odpowiada $|\mathrm{pull}|<2$.


In [ ]:
fig, axes = plt.subplots(nrows, ncols, figsize=(13, 3.6 * nrows), sharey=True)
axes = np.asarray(axes).ravel()

for ax, z in zip(axes, z_values):
    mask = np.isclose(dataset.z, z)
    order = np.argsort(dataset.k_velocity[mask])
    kv = dataset.k_velocity[mask][order]
    pull_z = pull[mask][order]

    ax.axhspan(-2, 2, color="tab:blue", alpha=0.09)
    ax.axhline(0, color="black", lw=1)
    ax.axhline(2, color="0.5", lw=0.8, ls="--")
    ax.axhline(-2, color="0.5", lw=0.8, ls="--")
    ax.plot(kv, pull_z, marker="o", ms=4, lw=1.3, color="tab:blue")
    ax.set_xscale("log")
    ax.set_xlabel(r"$k_v\,[\mathrm{s/km}]$")
    ax.set_ylabel(r"$(P_{\rm model}-P_{\rm data})/\sigma_{\rm total}$")
    ax.set_title(rf"$z={z:.1f}$")
    ax.grid(alpha=0.2)

for ax in axes[len(z_values):]:
    ax.set_visible(False)
fig.suptitle("Linear best-fit pulls", y=1.002)
fig.tight_layout()
if SAVE_FIGURES:
    fig.savefig(OUTPUT_DIR / "linear_best_fit_pulls.png", dpi=180, bbox_inches="tight")
plt.show()


## 11. Wspólny wykres z przesunięciami pionowymi

Ten wariant jest przydatny do szybkiego porównania kształtu widma pomiędzy redshiftami. Przesunięcia są wyłącznie wizualne i nie uczestniczą w ficie.


In [ ]:
fig, ax = plt.subplots(figsize=(10.5, 7.5))
colors = plt.cm.viridis(np.linspace(0.08, 0.92, len(z_values)))
offset_step = 0.035

for index, (z, color) in enumerate(zip(z_values, colors)):
    mask = np.isclose(dataset.z, z)
    order = np.argsort(dataset.k_velocity[mask])
    kv = dataset.k_velocity[mask][order]
    offset = index * offset_step
    data_y = plot_quantity(kv, dataset.p1d[mask][order])
    error_y = plot_quantity(kv, dataset.sigma_total[mask][order])
    model_y = plot_quantity(kv, model_best[mask][order])

    ax.errorbar(
        kv, data_y + offset, yerr=error_y, fmt="o", ms=3, capsize=1.5,
        color=color, alpha=0.75,
    )
    ax.plot(kv, model_y + offset, lw=2, color=color, label=rf"$z={z:.1f}$")

ax.set_xscale("log")
ax.set_xlabel(r"$k_v\,[\mathrm{s/km}]$")
ax.set_ylabel(r"$k_vP_{1D}(k_v)/\pi$ + vertical offset")
ax.set_title(rf"Linear best fit, $\chi^2={chi2_best:.3f}$")
ax.legend(ncol=2, frameon=False)
ax.grid(alpha=0.25)
fig.tight_layout()
if SAVE_FIGURES:
    fig.savefig(OUTPUT_DIR / "linear_best_fit_combined.png", dpi=180, bbox_inches="tight")
plt.show()


## 12. Diagnostyka redshiftu i ewolucja parametrów efektywnych

Wykres po lewej wskazuje, które bloki danych wnoszą najwięcej do całkowitego $\chi^2$. Linia na poziomie 35 oznacza liczbę punktów w bloku i jest jedynie orientacyjną referencją. Po prawej pokazujemy wynikającą z sześciu parametrów globalnych ewolucję amplitudy i parametru $\beta(z)$.


In [ ]:
log_alpha_f, beta_f, alpha_bias, beta_bias, alpha_ct, beta_ct = theta_best
redshift_ratio = (1 + z_values) / (1 + fitter.config.z_pivot)
nuisance_table = pd.DataFrame({
    "z": z_values,
    "A_F(z)": np.exp(log_alpha_f) * redshift_ratio**beta_f,
    "beta(z)": alpha_bias * redshift_ratio**beta_bias,
    "I_ct(z)": fitter.i0_scale * alpha_ct * redshift_ratio**(-beta_ct),
})
display(nuisance_table)

fig, axes_diag = plt.subplots(1, 2, figsize=(12.5, 4.5))
axes_diag[0].bar(chi2_by_z["z"].astype(str), chi2_by_z["chi2"], color="tab:blue")
axes_diag[0].axhline(35, color="black", ls="--", lw=1, label="35 data points")
axes_diag[0].set_xlabel("z")
axes_diag[0].set_ylabel(r"$\chi^2_z$")
axes_diag[0].set_title(r"Contribution to total $\chi^2$")
axes_diag[0].legend(frameon=False)

ax_beta = axes_diag[1].twinx()
line_a = axes_diag[1].plot(z_values, nuisance_table["A_F(z)"], "o-", color="tab:blue", label=r"$A_F(z)$")
line_b = ax_beta.plot(z_values, nuisance_table["beta(z)"], "s--", color="tab:orange", label=r"$\beta(z)$")
axes_diag[1].set_xlabel("z")
axes_diag[1].set_ylabel(r"$A_F(z)$", color="tab:blue")
ax_beta.set_ylabel(r"$\beta(z)$", color="tab:orange")
axes_diag[1].set_title("Effective nuisance evolution")
axes_diag[1].legend(line_a + line_b, [line.get_label() for line in line_a + line_b], frameon=False)
fig.tight_layout()
if SAVE_FIGURES:
    fig.savefig(OUTPUT_DIR / "linear_fit_diagnostics.png", dpi=180, bbox_inches="tight")
plt.show()


## 13. Zapis wyników walidacji

Zapisujemy tabelę walidacyjną, parametry, diagnostykę redshiftu oraz tablice danych i modelu. Dzięki temu późniejsze rysunki można tworzyć bez ponownego fitowania.


In [ ]:
validation_table.to_csv(OUTPUT_DIR / "validation_checks.csv", index=False)
parameter_table.to_csv(OUTPUT_DIR / "best_fit_parameters.csv", index=False)
chi2_diagnostics.to_csv(OUTPUT_DIR / "chi2_and_pulls_by_redshift.csv", index=False)
nuisance_table.to_csv(OUTPUT_DIR / "nuisance_evolution.csv", index=False)

np.savez_compressed(
    OUTPUT_DIR / "linear_best_fit_curves.npz",
    z=dataset.z,
    k_velocity=dataset.k_velocity,
    p1d_data=dataset.p1d,
    sigma_total=dataset.sigma_total,
    p1d_model=model_best,
    pull=pull,
    theta_best=theta_best,
    chi2=chi2_best,
)

validation_record = {
    "validation_passed": VALIDATION_PASSED,
    "published_chi2_target": PUBLISHED_CHI2,
    "published_tolerance": PUBLISHED_TOLERANCE,
    "fit": summary,
    "boundary_diagnostics": boundary_table.to_dict(orient="records"),
    "chi2_by_redshift": chi2_by_z.to_dict(orient="records"),
    "multistart": multistart_table.to_dict(orient="records"),
}
write_fit_result(OUTPUT_DIR / "linear_validation_fit.json", validation_record)

print("Saved validation outputs:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(" -", path.name)


## Interpretacja wyniku

Po przejściu wszystkich testów możemy uznać za zwalidowane: dane, jednostki, projekcję 3D→1D, liniowy model efektywny, kowariancję `paper_diag` i optymalizację sześciu parametrów. Nie waliduje to jeszcze generatora one-loop. Kolejnym krokiem, po otrzymaniu plików `production` i `precision` z klastra, będzie uruchomienie analogicznego notebooka porównującego linear i one-loop oraz skan $k_{\rm UV}=10,15,20\,h/\mathrm{Mpc}$.
